# NII-CU Multispectral Person Detection: 4-channel (RGB + thermal) YOLO

Recreates the early-fusion model from Speth et al. (2022), *Deep learning with RGB and thermal images onboard a drone for monitoring operations*.

**Pipeline**
1. Locate the dataset and parse the labels
2. Preprocess: filter labels, resize, stack RGB+thermal into 4-channel TIFFs, convert labels to YOLO format
3. Sanity-check the boxes visually
4. Train YOLO with `channels: 4`
5. Evaluate (AP@0.5, AP@0.75, AP@0.5:0.95) and compare to the paper

**Kaggle settings:** GPU on, Internet on (needed for `pip` and pretrained weights).

Paper targets (4-channel YOLOv3, aligned subset): AP@0.5 = 97.9, AP@0.75 = 76.9, AP@0.5:0.95 = 64.4.

In [ ]:
!pip install -q -U ultralytics
import ultralytics; ultralytics.checks()

## 1. Config and dataset discovery

In [ ]:
import os, glob, shutil, random
from pathlib import Path
import numpy as np, cv2

# ---------------- settings ----------------
ROOT_OVERRIDE = None          # e.g. "/kaggle/input/nii-cu-mapd/NII_CU_MAPD_dataset/4-channel"
LONG_SIDE     = 1280          # resize so the long image side = this (keeps disk use sane)
OUT           = Path("/kaggle/working/nii4ch")
KEEP_TYPE     = {0}           # 0 = person visible in both RGB and thermal
# ------------------------------------------

def find_root():
    if ROOT_OVERRIDE:
        return Path(ROOT_OVERRIDE)
    cands = []
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir() and (p / "images" / "rgb").is_dir() and (p / "images" / "thermal").is_dir() and (p / "labels").is_dir():
            cands.append(p)
    print("Candidate roots:", *cands, sep="\n  ")
    four = [c for c in cands if "4" in c.name and "channel" in c.name.lower()]
    if not four:
        raise RuntimeError("Could not find the 4-channel folder; set ROOT_OVERRIDE.")
    return four[0]

ROOT = find_root()
print("Using:", ROOT)
print("Splits:", [p.name for p in (ROOT/'images'/'rgb').iterdir()])
for sp in (ROOT/'images'/'rgb').iterdir():
    print(sp.name, len(list(sp.glob('*'))), "rgb images")

In [ ]:
# Peek at one sample
sp = sorted((ROOT/'images'/'rgb').iterdir())[0]
f = sorted(sp.glob('*'))[0]
print("RGB sample:", f)
img = cv2.imread(str(f)); print("RGB shape:", img.shape)
t = next((ROOT/'images'/'thermal'/sp.name).glob(f.stem + '.*'))
th = cv2.imread(str(t), cv2.IMREAD_UNCHANGED); print("Thermal:", t.name, th.shape, th.dtype)
if th.ndim == 3:
    print("Thermal channels identical (R=G=B)?", np.array_equal(th[...,0], th[...,1]) and np.array_equal(th[...,1], th[...,2]))
lab = next((ROOT/'labels'/sp.name).glob(f.stem + '.*'))
print("Label file:", lab.name); print(open(lab).read()[:300])

## 2. Preprocess

- Drop boxes with `bad != 0` or `type` not in `KEEP_TYPE`
- Resize RGB and thermal to the same size, store as a 4-plane TIFF in **R, G, B, T** order (uint8)
- Convert labels to normalized YOLO format (single class: person). Frames with no people get empty label files (negatives).

In [ ]:
def read_labels(path):
    rows = []
    if path is None: return rows
    for line in open(path):
        p = line.split()
        if len(p) < 7: continue
        try: rows.append([float(x) for x in p[:7]])
        except ValueError: continue   # skips a header line if present
    return rows

stats = dict(images=0, boxes_kept=0, boxes_dropped_type=0, boxes_dropped_bad=0, boxes_oob=0, empty_frames=0)

def convert(split):
    (OUT/'images'/split).mkdir(parents=True, exist_ok=True)
    (OUT/'labels'/split).mkdir(parents=True, exist_ok=True)
    rgb_files = sorted((ROOT/'images'/'rgb'/split).glob('*'))
    for i, rgb_path in enumerate(rgb_files):
        stem = rgb_path.stem
        th_path = next((ROOT/'images'/'thermal'/split).glob(stem + '.*'), None)
        if th_path is None:
            print("missing thermal for", stem); continue
        rgb = cv2.imread(str(rgb_path), cv2.IMREAD_COLOR)           # BGR
        th  = cv2.imread(str(th_path),  cv2.IMREAD_GRAYSCALE)
        if th.shape[:2] != rgb.shape[:2]:
            th = cv2.resize(th, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_LINEAR)
        h, w = rgb.shape[:2]
        s = LONG_SIDE / max(h, w)
        size = (round(w*s), round(h*s))
        rgb = cv2.resize(rgb, size, interpolation=cv2.INTER_AREA)
        th  = cv2.resize(th,  size, interpolation=cv2.INTER_AREA)
        b, g, r = cv2.split(rgb)
        planes = [r, g, b, th]                                      # RGB + T
        out_img = OUT/'images'/split/f"{stem}.tiff"
        ok = cv2.imwritemulti(str(out_img), planes, [cv2.IMWRITE_TIFF_COMPRESSION, 5])  # LZW
        if not ok:
            ok = cv2.imwritemulti(str(out_img), planes)
        assert ok, f"failed writing {out_img}"

        lab_path = next((ROOT/'labels'/split).glob(stem + '.*'), None)
        lines = []
        for x1, y1, x2, y2, typ, occ, bad in read_labels(lab_path):
            if bad != 0:               stats['boxes_dropped_bad'] += 1; continue
            if int(typ) not in KEEP_TYPE: stats['boxes_dropped_type'] += 1; continue
            if x1 < 0 or y1 < 0 or x2 > w or y2 > h: stats['boxes_oob'] += 1
            x1, x2 = np.clip([x1, x2], 0, w); y1, y2 = np.clip([y1, y2], 0, h)
            if x2 - x1 < 2 or y2 - y1 < 2: continue
            lines.append(f"0 {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
        (OUT/'labels'/split/f"{stem}.txt").write_text("\n".join(lines))
        stats['images'] += 1; stats['boxes_kept'] += len(lines)
        if not lines: stats['empty_frames'] += 1
        if (i+1) % 500 == 0: print(split, i+1, "/", len(rgb_files))

splits = sorted(p.name for p in (ROOT/'images'/'rgb').iterdir() if p.is_dir())
for sp in splits: convert(sp)
print(stats)
!du -sh /kaggle/working/nii4ch

**Check the stats above.** `boxes_oob` should be near zero. If it's large, the labels likely refer to the *uncropped* RGB frame and need to be shifted. The next cell will show it visually.

## 3. Sanity check: overlay the boxes

In [ ]:
import matplotlib.pyplot as plt

def load_tiff(p):
    ok, planes = cv2.imreadmulti(str(p), flags=cv2.IMREAD_UNCHANGED)
    assert ok
    return np.stack(planes, axis=2)           # H, W, 4  (R,G,B,T)

def draw(img, label_path, color):
    img = img.copy(); h, w = img.shape[:2]
    for ln in open(label_path).read().splitlines():
        _, cx, cy, bw, bh = map(float, ln.split())
        x1, y1 = int((cx-bw/2)*w), int((cy-bh/2)*h); x2, y2 = int((cx+bw/2)*w), int((cy+bh/2)*h)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    return img

val_imgs = sorted((OUT/'images'/splits[-1]).glob('*.tiff'))
sample = random.sample(val_imgs, 3)
fig, ax = plt.subplots(3, 3, figsize=(18, 11))
for r, p in enumerate(sample):
    a = load_tiff(p); lp = OUT/'labels'/p.parent.name/f"{p.stem}.txt"
    rgb = np.ascontiguousarray(a[..., :3]); th = np.ascontiguousarray(a[..., 3])
    blend = cv2.addWeighted(rgb, 0.6, cv2.cvtColor(th, cv2.COLOR_GRAY2RGB), 0.4, 0)
    ax[r,0].imshow(draw(rgb, lp, (0,0,255))); ax[r,0].set_title(p.stem + " RGB")
    ax[r,1].imshow(draw(cv2.cvtColor(th, cv2.COLOR_GRAY2RGB), lp, (0,255,0))); ax[r,1].set_title("thermal")
    ax[r,2].imshow(blend); ax[r,2].set_title("blend")
    for c in range(3): ax[r,c].axis('off')
plt.tight_layout(); plt.show()

If boxes sit on people in **both** the RGB and thermal panels, you're good. If they're shifted, stop and fix the label offset before training.

## 4. Dataset YAML

In [ ]:
yaml_text = f"""path: {OUT}
train: images/train
val: images/{splits[-1]}
channels: 4
names:
  0: person
"""
(OUT/'data.yaml').write_text(yaml_text)
print(yaml_text)

## 5. Train

The pretrained weights are 3-channel, so Ultralytics skips the mismatched first conv layer. A callback fixes that the way the paper does: copy the pretrained RGB kernels, and initialize the thermal kernel from the **red-channel** kernel.

`MODEL`: `yolo11s.pt` is a good fit for Kaggle GPUs. For a closer reproduction use `yolov3u.pt` (Ultralytics' YOLOv3), but it is much heavier at 1280px.

In [ ]:
import torch
from ultralytics import YOLO

MODEL  = "yolo11s.pt"      # or "yolov3u.pt"
EPOCHS = 50                # paper: 50
IMGSZ  = 1280
BATCH  = 8                 # lower to 4 if you hit OOM

# keep the pretrained RGB first-layer weights to seed the new 4-channel layer
pre = YOLO(MODEL)
w_pre = pre.model.model[0].conv.weight.detach().clone().cpu()     # (out, 3, k, k)
del pre

def init_4ch_first_layer(trainer):
    def patch(m):
        conv = m.model[0].conv
        assert conv.weight.shape[1] == 4, "model is not 4-channel; check data.yaml channels"
        with torch.no_grad():
            w = w_pre.to(conv.weight.device, conv.weight.dtype)
            conv.weight[:, :3] = w
            conv.weight[:, 3:4] = w[:, 0:1]          # thermal <- red kernel (as in the paper)
    patch(trainer.model)
    if getattr(trainer, "ema", None) is not None:
        patch(trainer.ema.ema)
    print("Initialized 4-channel first conv layer from pretrained RGB weights")

model = YOLO(MODEL)
model.add_callback("on_train_start", init_4ch_first_layer)

results = model.train(
    data=str(OUT/'data.yaml'),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    optimizer="SGD", lr0=0.01, cos_lr=False,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,     # colour jitter makes no sense with a thermal channel
    flipud=0.0, fliplr=0.5, mosaic=1.0, close_mosaic=10,
    workers=2, cache=False, patience=20,
    project="/kaggle/working/runs", name="nii_cu_4ch", exist_ok=True,
)

## 6. Evaluate

In [ ]:
best = YOLO("/kaggle/working/runs/nii_cu_4ch/weights/best.pt")
m = best.val(data=str(OUT/'data.yaml'), imgsz=IMGSZ, batch=BATCH, conf=0.001, iou=0.5)
print(f"AP@0.5      {m.box.map50*100:.1f}   (paper 97.9)")
print(f"AP@0.75     {m.box.map75*100:.1f}   (paper 76.9)")
print(f"AP@0.5:0.95 {m.box.map*100:.1f}   (paper 64.4)")

In [ ]:
# quick qualitative check on a few val images
imgs = [str(p) for p in random.sample(val_imgs, 4)]
for r in best.predict(imgs, imgsz=IMGSZ, conf=0.25, save=True, project="/kaggle/working/runs", name="preds", exist_ok=True):
    pass
print("Saved to /kaggle/working/runs/preds")

## Notes
- **Val is the paper's test set** (the 4-channel `val` folder, ~485 images). It comes from the same site as training, so expect optimistic numbers.
- Different model/optimizer/resolution than the paper's YOLOv3 code means results won't match exactly; being in the same range (high-90s AP@0.5) is a good sign.
- **Ablations to run next:** train RGB-only and thermal-only (swap in `channels: 3` / `channels: 1` with the matching images) to reproduce the paper's comparison table.
- If disk fills up, lower `LONG_SIDE` (e.g. 1024).